In [ ]:
import lightkurve
from lightkurve import DesignMatrix, RegressionCorrector
import matplotlib.pyplot as plot
import numpy
%matplotlib inline
plot.style.use('finny_style.mplstyle')

In [ ]:
star_id = 295440174
star_id = f'TIC {star_id}'

In [ ]:
search_result = lightkurve.search_targetpixelfile(star_id, mission = 'TESS', cadence = 'long')
search_result

In [ ]:
pixel_file = search_result[-1].download()
pixel_file = pixel_file[numpy.isfinite(pixel_file.flux).all(axis = (1, 2))]
pixel_file

In [ ]:
#for pixel_file in pixel_files:
aperture_mask = pixel_file.pipeline_mask
pixel_file.plot(aperture_mask = aperture_mask)

plot.show()

In [ ]:
uncorrected_lc = pixel_file.to_lightcurve(aperture_mask = aperture_mask)
uncorrected_lc.plot()
plot.show()

In [ ]:
regressors = pixel_file.flux[:, ~aperture_mask]
regressors.shape

In [ ]:
plot.plot(regressors[:, :30])
plot.show()

In [ ]:
design_matrix = DesignMatrix(pixel_file.flux[:, ~aperture_mask], name = f'{star_id} regressors')
design_matrix

In [ ]:
design_matrix = design_matrix.pca(5)
design_matrix = design_matrix.append_constant()
design_matrix

In [ ]:
plot.plot(pixel_file.time.value, design_matrix.values[:, :5] + (numpy.arange(5) * 0.2))
plot.show()

In [ ]:
corrector = RegressionCorrector(uncorrected_lc)
corrector

In [ ]:
lc = corrector.correct(design_matrix)
lc

In [ ]:
ax = uncorrected_lc.plot(label = 'original')
lc.plot(ax = ax, label = 'new')
plot.show()

In [ ]:
corrector.diagnose()
plot.show()

In [ ]:
_lc = lc.flatten(break_tolerance = 10, niters = 10)#.remove_outliers(sigma = 10.0)
_lc.plot()
plot.show()

In [ ]:
pixel_files = search_result.download_all()

In [ ]:
collection = lightkurve.LightCurveCollection(None)
for pixel_file in pixel_files[:]:
    pixel_file = pixel_file[numpy.isfinite(pixel_file.flux).all(axis = (1, 2))]
    pixel_file = pixel_file[numpy.isfinite(pixel_file.flux_err).all(axis = (1, 2))]

    aperture_mask = pixel_file.pipeline_mask
    # aperture_mask = pixel_file.create_threshold_mask(threshold = 10, reference_pixel = 'center')
    uncorrected_lc = pixel_file.to_lightcurve(aperture_mask = aperture_mask)

    design_matrix = DesignMatrix(pixel_file.flux[:, ~aperture_mask]).pca(5).append_constant()
    lc = RegressionCorrector(uncorrected_lc).correct(design_matrix)
    # lc = uncorrected_lc
    lc = lc.normalize().flatten(niters = 10).remove_outliers(10.0)
    lc.plot()
    collection.append(lc)

plot.show()

In [ ]:
lc = collection[:].stitch()
lc = lc.flatten(break_tolerance = 10, niters = 10).remove_outliers(sigma = 10.0)
lc.plot()
plot.show()

In [ ]:
periods = numpy.logspace(0, 1.5, 10 ** 4, base = 10.0)
periodogram = lc.to_periodogram(method = 'boxleastsquares', period = periods, frequency_factor = 10 ** 6)
periodogram.plot()
plot.show()

In [ ]:
period = periodogram.period_at_max_power
transit_time = periodogram.transit_time_at_max_power
transit_duration = periodogram.duration_at_max_power
transit_duration = 1

period, transit_time, transit_duration

In [ ]:
folded_lc = lc.fold(period = period, epoch_time = transit_time)
binned_lc = folded_lc.bin(time_bin_size = 0.01)
ax = folded_lc.scatter()
binned_lc.plot(ax = ax, c = 'r')
d = 1
# ax.set_xlim(-d, d)
# ax.set_ylim(0.995, 1.0025)
plot.show()
1 - numpy.nanmin(binned_lc.flux.value)

In [ ]:
pixel_file = pixel_files[0]
pixel_file.plot_pixels(aperture_mask = pixel_file.pipeline_mask)
plot.show()

In [ ]:
mask = periodogram.get_transit_mask(
            period = period,
            transit_time = transit_time,
            duration = transit_duration
        )

lc_mask = lc[mask]
ax = lc_mask.scatter()
lc = lc[~mask]
lc.scatter(ax = ax)
plot.show()